In [2]:
import numpy as np
import pandas as pd
import re
import torch
import torch.nn as nn
import torch.nn.functional as F

class NN(nn.Module):
    def __init__(self, num_classes):
        super(NN, self).__init__()

        self.embedding = nn.Embedding(27, 16)
        self.fc1 = nn.Linear(16*50, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, num_classes)

    def forward(self, x):
        
        x = x.long()
        x = self.embedding(x)
        x = x.flatten(1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = self.fc4(x)
        return x 

GPU_indx = 0
device = torch.device(GPU_indx if torch.cuda.is_available() else 'cpu')

model = NN(27).to(device)
model.load_state_dict(torch.load("model.pth"))
model.eval()


NN(
  (embedding): Embedding(27, 16)
  (fc1): Linear(in_features=800, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=256, bias=True)
  (fc3): Linear(in_features=256, out_features=128, bias=True)
  (fc4): Linear(in_features=128, out_features=27, bias=True)
)

In [14]:
def create_caesar(s: str, p):
    
    t = 'abcdefghijklmnopqrstuvwxyz '
    s = s.lower()
    s = re.sub(r'[^a-z\s]', '', s)
    s = list(s)
    s = s[:50]
    # print(''.join(s))
    for i in range(len(s)):
        if s[i] == ' ': 
            s[i] = ((ord(s[i])-32+26)+p)%27
        else:
            s[i] = ((ord(s[i])-97)+p)%27
    for i in range(len(s)):
        s[i] = t[s[i]]
    s = ''.join(s)    
    # print(s)
    return s

In [4]:
def test_model(s: str):
    t = 'abcdefghijklmnopqrstuvwxyz '
    s = s.lower()
    s = re.sub(r'[^a-z\s]', '', s)
    s = list(s)
    s = s[:50]
    for i in range(len(s)):
        if s[i] == ' ': 
            s[i] = (ord(s[i])-32+26)
        else:
            s[i] = (ord(s[i])-97)
    
    x = torch.tensor(s).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(x)
        pred = logits.argmax(dim=1)

    shift = pred.item()

    for i in range(len(s)):
        s[i] = (s[i]+27-shift)%27

    for i in range(len(s)):
        s[i] = t[s[i]]
    
    s = ''.join(s)
    print(s)

In [15]:
s = "mind, That have their first conception by misdread, Have after-nourishment and life by care; And what was first but fear what might be done, Grows elder now and cares it be not"

In [16]:
d = create_caesar(s, 16) # 26 here is the rotation index
print(d)

byctpixqipxqkupixuygpvyghipsdcsueiydcprnpbyhtguqtp


In [17]:
test_model(d)

mind that have their first conception by misdread 
